#### **Packages** ####

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy

#### **Recovery Coefficient (RC) Calculation** ####

In [3]:
def get_data_rc(data_bqml, sphere_concentration):
    """ This function receives the CSV file containing the VOI statistics obtained by application of the 
    quantitative NEMA workflow developed within the context of this manuscript - data_bqml - and the
    sphere activity concentration corrected for time-decay - sphere_concentration - and computes the RCs 
    for each NEMA spheres. It returns a dataframe containing the volume of each NEMA sphere VOI (in mL),
    the total activity within each NEMA sphere (in MBq) and the corresponding NEMA sphere RC. """
    spheres = {}
    for i in range(1, 7):
        sphere_bq = getattr(data_bqml, f'Sphere- {i}')
        bg_sphere_bq = getattr(data_bqml, f'BG_Sphere {i}')
        sphere_mbq_final = (float(sphere_bq[1]) - float(bg_sphere_bq[1]))*10**(-6)
        volume = float(sphere_bq[6])
        true_activity = sphere_concentration*volume
        rc = sphere_mbq_final/true_activity
        spheres[f'sph{i}'] = {'MBq': sphere_mbq_final, 'vol': volume, 'RC': rc}
    mbq_data = (spheres['sph1']['MBq'], spheres['sph2']['MBq'], spheres['sph3']['MBq'], spheres['sph4']['MBq'], spheres['sph5']['MBq'], spheres['sph6']['MBq'])
    volume_data = (spheres['sph1']['vol'], spheres['sph2']['vol'], spheres['sph3']['vol'], spheres['sph4']['vol'], spheres['sph5']['vol'], spheres['sph6']['vol'])
    rc_data = (spheres['sph1']['RC'], spheres['sph2']['RC'], spheres['sph3']['RC'], spheres['sph4']['RC'], spheres['sph5']['RC'], spheres['sph6']['RC'])
    data = pd.DataFrame({'Volume': volume_data, 'Activity': mbq_data, 'RC': rc_data})
    return data

In [5]:
# This code cell creates a dataframe with all RC values for all the StarGuide/Q.Clear reconstructions. 
# For Symbia/OSEM reconstruction, the name of the file (bqml_file) should be changed accordingly.
# For 177Lu protocols, the name of the file should also be changed to match the CSV files. 
"""
temp_master_dfs = []
r = 'Number of Reconstructions'
concentration = 'Sphere Concentration'
for i in range(0, r):
    recon = i + 1
    bqml_file = f'./RR{recon}.csv' # VOI Statistics File
    data_bqml = pd.read_csv(bqml_file)
    data = get_data_rc(data_bqml, concentration)
    temp_master_df = pd.DataFrame({f'Volume': data.Volume.values, 'RC': data.RC.values})
    temp_master_df.to_csv(f'./RR{recon}_RC.csv')
    temp_master_dfs.append(temp_master_df)
master_df = pd.concat(temp_master_dfs, axis = 1)
master_df.to_csv('data_RC.csv', index = False)
"""

"\ntemp_master_dfs = []\nr = 'Number of Reconstructions'\nconcentration = 'Sphere Concentration'\nfor i in range(0, r):\n    recon = i + 1\n    bqml_file = f'./RR{recon}.csv' # VOI Statistics File\n    data_bqml = pd.read_csv(bqml_file)\n    data = get_data_rc(data_bqml, concentration)\n    temp_master_df = pd.DataFrame({f'Volume': data.Volume.values, 'RC': data.RC.values})\n    temp_master_df.to_csv(f'Data_RC/RR{recon}_RC.csv')\n    temp_master_dfs.append(temp_master_df)\n\nmaster_df = pd.concat(temp_master_dfs, axis = 1)\nmaster_df.to_csv('data_RC.csv', index = False)\n"

#### **Background Variability (BV) Calculation** ####

In [6]:
def get_data_bv(recon):
    """ This function retrieves the CSV file containing the VOI statistics for each 25 VOI set used for 
    BV calculation and computes BV for each VOI volume. The name of the file should be changed according to the VOI statistics
    CSV file. """
    bv_data = []
    list_sphere = [5, 4, 3, 2, 1]
    spheres = {'SB1': 100, 'SB2': 50, 'SB3': 26.5, 'SB4': 2.6, 'SB5': 0.5}
    for i in range(0,5):
        sphere = list_sphere[i]
        bqml_file = f'./RR{recon}_{sphere}.csv'
        data_bqml = pd.read_csv(bqml_file)
        mean_row = data_bqml[data_bqml.iloc[:, 0].str.contains('Mean', na = False)]
        bg_mean = mean_row.iloc[0, 1:].values.astype(float)
        mean = np.mean(bg_mean)
        std = np.std(bg_mean)
        bv = (std/mean)*100
        tick = f'SB{sphere}'
        bv_data.append({'Volume': spheres.get(tick, 'N/A'), 'Sphere': f'SB{sphere}', 
        'Mean BG': mean, 'SD': std, 'BV': bv})
        BV = pd.DataFrame(bv_data)
    return BV

In [10]:
# This code cell creates a dataframe with all BV values for all the StarGuide/Q.Clear reconstructions. 
# For Symbia/OSEM reconstruction, the name of the file (bqml_file) should be changed accordingly.
# For 177Lu protocols, the name of the file should also be changed to match the CSV files. 
"""
temp_master_dfs = []
recons = np.arange(., ., .)
for recon in recons:
    data =  get_data_bv(recon)
    temp_master_df = pd.DataFrame({f'Volume': data.Volume.values, 'BV': data.BV.values})
    temp_master_df.to_csv(f'./RR{recon}_BV.csv')
    temp_master_dfs.append(temp_master_df)
    plt.plot(data.Volume.values, data.BV.values, '--', marker = 'o', markersize = 3, label = f'RR{recon}')
    # plt.show()
master_df = pd.concat(temp_master_dfs, axis = 1)
master_df.to_csv('data_BV.csv', index = False)
"""

"\ntemp_master_dfs = []\nrecons = np.arange(., ., .)\nfor recon in recons:\n    data =  get_data_bv(recon)\n    temp_master_df = pd.DataFrame({f'Volume': data.Volume.values, 'BV': data.BV.values})\n    temp_master_df.to_csv(f'./RR{recon}_BV.csv')\n    temp_master_dfs.append(temp_master_df)\n    plt.plot(data.Volume.values, data.BV.values, '--', marker = 'o', markersize = 3, label = f'RR{recon}')\n    # plt.show()\nmaster_df = pd.concat(temp_master_dfs, axis = 1)\nmaster_df.to_csv('data_BV.csv', index = False)\n"

#### **Signal-to-Noise Ratio** ###

In [11]:
def get_data_snr(data_bqml):
    """ This function receives the CSV file containing the VOI statistics obtained by application of the 
    quantitative NEMA workflow developed within the context of this manuscript - data_bqml -, and computes the SNR for each NEMA spheres. 
    It returns a dataframe containing the volume of each NEMA sphere VOI (in mL) and the SNR associated with each sphere."""    
    spheres = {}
    for i in range(1, 7):
        sphere_bq = getattr(data_bqml, f'Sphere- {i}')
        bg_sphere_bq = getattr(data_bqml, f'BG_Sphere {i}')
        volume = float(sphere_bq[6])
        snr = (float(sphere_bq[3])-float(bg_sphere_bq[3]))/float(bg_sphere_bq[4])
        spheres[f'sph{i}'] = {'vol': volume, 'SNR': snr}
    volume_data = (spheres['sph1']['vol'], spheres['sph2']['vol'], spheres['sph3']['vol'], spheres['sph4']['vol'], spheres['sph5']['vol'], spheres['sph6']['vol'])
    snr_data = (spheres['sph1']['SNR'], spheres['sph2']['SNR'], spheres['sph3']['SNR'], spheres['sph4']['SNR'], spheres['sph5']['SNR'], spheres['sph6']['SNR'])
    data = pd.DataFrame({'Volume': volume_data, 'SNR': snr_data})
    return data

In [ ]:
# This code cell creates a dataframe with all CRC values for all the StarGuide/Q.Clear reconstructions. 
# For Symbia/OSEM reconstruction, the name of the file (bqml_file) should be changed accordingly.
# For 177Lu protocols, the name of the file should also be changed to match the CSV files. 
"""
temp_master_dfs = []
r = 'Number of Reconstructions'
for i in range(0, r):
    recon = i + 1
    bqml_file = f'./RR{recon}.csv'
    data_bqml = pd.read_csv(bqml_file)
    data =  get_data_snr(data_bqml)
    temp_master_df = pd.DataFrame({f'Volume': data.Volume.values, 'SNR': data.SNR.values})
    temp_master_df.to_csv(f'./RR{recon}_SNR.csv')
    temp_master_dfs.append(temp_master_df)
    plt.plot(data.Volume, data.SNR, 'o:', color = 'black', linewidth = 1.8, label = f'SNR_RR{recon}', markersize = 4)
    plt.xlabel('Volume (ml)')
    plt.ylabel('SNR', alpha = 0.75)
    plt.grid(True)
    plt.legend()
    plt.show()
master_df = pd.concat(temp_master_dfs, axis = 1)
master_df.to_csv('data_SNR.csv', index = False)
"""

#### **Contrast Recovery Coefficient (CRC)** ####

In [8]:
def get_data_contrast(data_bqml, sphere_concentration, bg_concentration):
    """ This function receives the CSV file containing the VOI statistics obtained by application of the 
    quantitative NEMA workflow developed within the context of this manuscript - data_bqml -, the
    sphere activity concentration corrected for time-decay - sphere_concentration -, and the phantom's background
    concentration corrected for time-decay - bg_concentration -, and computes the CRC for each NEMA spheres. 
    It returns a dataframe containing the volume of each NEMA sphere VOI (in mL) and the CRC associated 
    with each sphere."""
    spheres = {}
    for i in range(1, 7):
        sphere_bq = getattr(data_bqml, f'Sphere- {i}')
        bg_sphere_bq = getattr(data_bqml, f'BG_Sphere {i}')
        true_activity_sphere = sphere_concentration
        true_activity_bg = bg_concentration
        volume = sphere_bq[6]
        sphere_mbq = float(sphere_bq[3])*10**(-6)
        bg_sphere_mbq = float(bg_sphere_bq[3])*10**(-6)
        contrast = (((float(sphere_mbq)/float(bg_sphere_mbq))-1)/((true_activity_sphere/true_activity_bg)-1))*100
        spheres[f'sph{i}'] = {'vol': volume, 'contrast': contrast}
    cont_pecontrast_data = (spheres['sph1']['contrast'], spheres['sph2']['contrast'], spheres['sph3']['contrast'], spheres['sph4']['contrast'], spheres['sph5']['contrast'], spheres['sph6']['contrast'])
    volume_data = (spheres['sph1']['vol'], spheres['sph2']['vol'], spheres['sph3']['vol'], spheres['sph4']['vol'], spheres['sph5']['vol'], spheres['sph6']['vol'])
    data = pd.DataFrame({'Volume': volume_data, 'Contrast': cont_pecontrast_data})
    return data

In [9]:
# This code cell creates a dataframe with all CRC values for all the StarGuide/Q.Clear reconstructions. 
# For Symbia/OSEM reconstruction, the name of the file (bqml_file) should be changed accordingly.
# For 177Lu protocols, the name of the file should also be changed to match the CSV files. 
"""
temp_master_dfs = []
r = 20
for i in range(0, r):
    recon = i + 1
    bqml_file = pd.read_csv(f'RR/RR{recon}.csv')
    data =  get_data_contrast(bqml_file, sphere_concentration, bg_concentration)
    temp_master_df = pd.DataFrame({f'Volume': data.Volume.values, 'Contrast': data.Contrast.values})
    temp_master_df.to_csv(f'./RR{recon}_Contrast.csv')
    temp_master_dfs.append(temp_master_df)
master_df = pd.concat(temp_master_dfs, axis = 1)
master_df.to_csv('data_Contrast.csv', index = False)
"""

"\ntemp_master_dfs = []\nr = 20\nfor i in range(0, r):\n    recon = i + 1\n    bqml_file = pd.read_csv(f'RR/RR{recon}.csv')\n    data =  get_data_contrast(bqml_file, sphere_concentration, bg_concentration)\n    temp_master_df = pd.DataFrame({f'Volume': data.Volume.values, 'Contrast': data.Contrast.values})\n    temp_master_df.to_csv(f'./RR{recon}_Contrast.csv')\n    temp_master_dfs.append(temp_master_df)\nmaster_df = pd.concat(temp_master_dfs, axis = 1)\nmaster_df.to_csv('data_Contrast.csv', index = False)\n"

#### **EANM Fit** ####

In [ ]:
def EANM_fit(volume, a1, b1):
    """EANM recovery coefficient fitting function."""
    return 1 - (1 / (1 + (volume / a1)**b1))

def dEANM_a1(volume, a1, b1):
    """Partial derivative of the EANM fit with respect to a1."""
    return -(b1 * volume**b1) / (
        a1**(b1 + 1) * (1 + (volume / a1)**b1)**2)

def dEANM_b1(volume, a1, b1):
    """Partial derivative of the EANM fit with respect to b1."""
    x = volume / a1
    return (x**b1 * np.log(x)) / (1 + x**b1)**2

def get_r_squared(rc, p):
    """Calculates the coefficient of determination (R²)."""
    r = 1 - np.sum((rc - p)**2) / np.sum((rc - np.mean(rc))**2)
    return r

def get_EANM_fit(volume, rc, recon):
    """
    Fits the EANM RC model to the measured data, calculates R²,
    estimates parameter-based uncertainty bounds, and plots the
    fitted curve with the measured RC values.

    Parameters:
    volume : array-like
        Sphere volumes (mL).
    rc : array-like
        Measured recovery coefficients.
    recon : str or int
        Reconstruction identifier used for plot labelling and filename.

    Returns:
    v : numpy.ndarray
        Volume values used for the fitted curve.
    fit_EANM : numpy.ndarray
        Fitted EANM RC values.
    upper_bound : numpy.ndarray
        Upper 95% confidence band bound.
    lower_bound : numpy.ndarray
        Lower 95% confidence band bound.
    """
    p0 = [((rc[4]+rc[5])/2)/2, rc[2]]    
    param,pcov = scipy.optimize.curve_fit(EANM_fit, volume, rc, p0)
    perr = np.sqrt(np.diag(pcov))
    a1, b1 = param[0],param[1]
    v = np.arange(0.1, volume[5]*1.5, 0.01)
    fit_EANM = EANM_fit(v, a1, b1)
    p = EANM_fit(volume, a1, b1)
    r = get_r_squared(rc, p)
    plt.plot(v, fit_EANM, linestyle = '-', color = 'red', linewidth = 1, label = fr'EANM Fit: $R^2$ = {r:.3f}')
    plt.plot(volume, rc, '.', color = 'black', label = f'Data R{recon}')
    plt.xlim(0, volume[5]*1.5)
    plt.xlabel('Volume (ml)')
    plt.ylabel('Recovery Coefficient (RC)')
    plt.grid(True)
    r_list.append(r)
    y_err = np.sqrt((dEANM_a1(v,a1,b1)*perr[0])**2 + (dEANM_b1(v,a1,b1)*perr[1])**2)
    upper_bound = fit_EANM + y_err
    lower_bound = fit_EANM - y_err
    plt.plot(v, upper_bound, linestyle = '--', color = 'red', linewidth = 0.5)
    plt.plot(v, lower_bound, linestyle = '--', color = 'red', linewidth = 0.5)
    plt.fill_between(v, upper_bound, lower_bound, color = 'red', alpha = 0.1)
    plt.legend(loc = 'lower right', edgecolor = 'black', fancybox = True, facecolor = 'white')
    plt.savefig(f'graphs_thesis/FIT_R{recon}')
    plt.show()
    return v, fit_EANM, upper_bound, lower_bound